# Face Swap — Per-Region Contrastive Training

**Architecture:**
- 4 independent region encoders (mouth, eyes, ears, nose)
- Each: ResNet-50 (7ch: RGB+normal+depth) + Transformer
- Trained with NT-Xent contrastive loss on VGGFace2 triplets

**Training phases:**
1. Phase 1: Train each region encoder independently (this notebook)
2. Phase 2: Fine-tune with SDXL U-Net (separate notebook after Phase 1)

**Before running:**
1. Enable GPU: Settings -> Accelerator -> GPU T4 x2
2. Add VGGFace2 dataset (search 'vggface2' on Kaggle Datasets)
3. The code is cloned from GitHub automatically

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────
REPO_URL        = "https://github.com/raghavujjwal/Geometry-aware-Deepfake-generation-via-3D-disentanglement.git"
VGGFACE2_ROOT   = "/kaggle/input/vggface2/data/vggface2_train/train"
CHECKPOINT_DIR  = "/kaggle/working/checkpoints"
CODE_DIR        = "/kaggle/working/repo"

MAX_IDENTITIES  = 1000    # out of 9131 total — fits in Kaggle session
BATCH_SIZE      = 32      # per region encoder (small model, large batch ok)
EPOCHS          = 20      # per region (~1.5-2 hrs per region on T4)
LR              = 1e-4
TOKEN_DIM       = 512
TEMPERATURE     = 0.07    # NT-Xent temperature

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('Config OK')

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────
!pip install -q transformers scikit-image kornia

In [ ]:
# ── Clone repo and set paths ───────────────────────────────────────────────
!git clone {REPO_URL} {CODE_DIR}

import sys
sys.path.insert(0, f'{CODE_DIR}/kaggle')
sys.path.insert(0, f'{CODE_DIR}/kaggle/face_swap')
print('Repo cloned and paths set')

In [ ]:
# ── Verify GPU and imports ─────────────────────────────────────────────────
import torch
import torch.nn as nn

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

from face_swap.models.region_encoder import RegionEncoder, FaceRegionEncoder
from face_swap.data.vggface2_dataset import VGGFace2TripletDataset, collate_triplets
from face_swap.training.contrastive_train import NTXentLoss
print('Imports OK')

In [ ]:
# ── Preview dataset ────────────────────────────────────────────────────────
from torch.utils.data import DataLoader

# Quick check on mouth region
dataset = VGGFace2TripletDataset(
    root=VGGFACE2_ROOT,
    target_region='mouth',
    max_identities=10,    # just 10 for preview
    min_images=5,
)
sample = dataset[0]
print('anchor  mouth shape:', sample['anchor']['mouth'].shape)    # (7, 64, 64)
print('positive mouth shape:', sample['positive']['mouth'].shape)  # (7, 64, 64)
print('negative mouth shape:', sample['negative']['mouth'].shape)  # (7, 64, 64)
print('channel breakdown: RGB(0:3) | normal(3:6) | depth(6:7)')
del dataset

In [ ]:
# ── Train mouth encoder ────────────────────────────────────────────────────
from face_swap.training.contrastive_train import train_region

train_region(
    region='mouth',
    data_root=VGGFACE2_ROOT,
    checkpoint_dir=CHECKPOINT_DIR,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    max_identities=MAX_IDENTITIES,
    token_dim=TOKEN_DIM,
    temperature=TEMPERATURE,
    num_workers=2,
    device=DEVICE,
)

In [ ]:
# ── Train eyes encoder ─────────────────────────────────────────────────────
train_region(
    region='eyes',
    data_root=VGGFACE2_ROOT,
    checkpoint_dir=CHECKPOINT_DIR,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    max_identities=MAX_IDENTITIES,
    token_dim=TOKEN_DIM,
    temperature=TEMPERATURE,
    num_workers=2,
    device=DEVICE,
)

In [ ]:
# ── Train ears encoder ─────────────────────────────────────────────────────
train_region(
    region='ears',
    data_root=VGGFACE2_ROOT,
    checkpoint_dir=CHECKPOINT_DIR,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    max_identities=MAX_IDENTITIES,
    token_dim=TOKEN_DIM,
    temperature=TEMPERATURE,
    num_workers=2,
    device=DEVICE,
)

In [ ]:
# ── Train nose encoder ─────────────────────────────────────────────────────
train_region(
    region='nose',
    data_root=VGGFACE2_ROOT,
    checkpoint_dir=CHECKPOINT_DIR,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    max_identities=MAX_IDENTITIES,
    token_dim=TOKEN_DIM,
    temperature=TEMPERATURE,
    num_workers=2,
    device=DEVICE,
)

In [ ]:
# ── Verify all checkpoints saved ──────────────────────────────────────────
import os
ckpts = sorted(os.listdir(CHECKPOINT_DIR))
print(f'Checkpoints saved ({len(ckpts)} files):')
for f in ckpts:
    size = os.path.getsize(f'{CHECKPOINT_DIR}/{f}') / 1e6
    print(f'  {f}  ({size:.1f} MB)')

In [ ]:
# ── Quick sanity check: load best checkpoints and run inference ────────────
import torch
from face_swap.models.region_encoder import RegionEncoder

regions = ['mouth', 'eyes', 'ears', 'nose']
encoders = {}

for region in regions:
    best_ckpt = f'{CHECKPOINT_DIR}/{region}_best.pt'
    if not os.path.exists(best_ckpt):
        print(f'  {region}: no checkpoint found, skipping')
        continue
    enc = RegionEncoder(region_name=region, token_dim=TOKEN_DIM).to(DEVICE)
    enc.load_state_dict(torch.load(best_ckpt, map_location=DEVICE)['encoder'])
    enc.eval()
    encoders[region] = enc

    # Test forward pass
    dummy = torch.randn(2, 7, 64, 64).to(DEVICE)
    with torch.no_grad():
        token = enc(dummy)  # (2, 1, 512)
    print(f'  {region}: token shape={token.shape}  OK')

print('\nPhase 1 complete. Next: Phase 2 — U-Net fine-tuning with these frozen encoders.')